## Naive RAG over KG triples (line by line)

This notebook indexes **each JSONL line** from jsonl file created previously in the `kg_linearization.ipynb` as a separate document in Chroma, using Ollama embeddings (`nomic-embed-text`).

Pros: simplest possible baseline.
Cons: each triple is tiny, so retrieval can miss supporting facts that belong together. Needs lots of chubnks retrieved at once, and possible contextual limits of a line

In [ ]:
# Install dependencies (run once)
!pip install -q chromadb requests

In [ ]:
from pathlib import Path
import json
from typing import List, Dict, Any, Iterable, Optional

import requests
import chromadb
from chromadb.config import Settings

project_root = Path('.').resolve()
jsonl_path = project_root / 'Bundesliga23_24_test_triples_linearized.jsonl'


Project root: C:\Users\dyury\Desktop\Master Thesis
JSONL exists: True C:\Users\dyury\Desktop\Master Thesis\Bundesliga23_24_test_triples_linearized.jsonl


In [ ]:
def iter_jsonl_texts(path: Path, max_lines: Optional[int] = 50000) -> Iterable[str]:
    # Yield the 'text' field from each JSONL line.
    n = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if max_lines is not None and n >= max_lines:
                break
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            text = obj.get('text')
            if isinstance(text, str) and text:
                yield text
                n += 1


# Preview
preview = list(iter_jsonl_texts(jsonl_path, max_lines=5))
for i, t in enumerate(preview):
    print(i, t)


0 event/62ead0ab-e3ec-46fa-ab33-7b9d0473a344 x 48.6
1 match/3895052 events event/ce261d4b-9c4d-4414-a519-e95dea8a0fa3
2 event/7d2cc416-583e-4918-a35e-429b12a4aa83 y_end 70.0
3 event/d7f33026-3427-45b8-b838-3801ffb99a25 receiver_time 01:19:16.477
4 event/a8aa1263-3b1e-4640-89e7-434c09d27f53 event_period second_half


In [ ]:
class OllamaEmbeddingFunction:
    # Same minimal embedding function wrapper for Chroma using Ollama.

    def __init__(self, model: str = 'nomic-embed-text', base_url: str = 'http://localhost:11434'):
        self.model = model
        self.url = f'{base_url}/api/embeddings'

    def _embed_texts(self, texts: List[str]) -> List[List[float]]:
        vectors: List[List[float]] = []
        for text in texts:
            resp = requests.post(self.url, json={'model': self.model, 'prompt': text})
            resp.raise_for_status()
            vectors.append(resp.json()['embedding'])
        return vectors

    def embed_documents(self, input: List[str]) -> List[List[float]]:
        return self._embed_texts(input)

    def embed_query(self, input):
        if isinstance(input, str):
            return self._embed_texts([input])[0]
        return self._embed_texts(input)

    def __call__(self, input: List[str]) -> List[List[float]]:
        return self.embed_documents(input)

    def name(self) -> str:
        return f'ollama-{self.model}'


In [ ]:
# Build documents
MAX_LINES = 50000  # set None for full file, but if fails for my setting

docs: List[Dict[str, Any]] = []
for idx, text in enumerate(iter_jsonl_texts(jsonl_path, max_lines=MAX_LINES)):
    docs.append({'id': f'line::{idx}', 'text': text})

print('Docs:', len(docs))
print('Example:', docs[0])


Docs: 46725
Example: {'id': 'line::0', 'text': 'event/62ead0ab-e3ec-46fa-ab33-7b9d0473a344 x 48.6'}


In [6]:
# Create Chroma collection (batched add)
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
embedding_function = OllamaEmbeddingFunction()

COLLECTION_NAME = 'kg_triples_linewise'
RESET_COLLECTION = True  # set False to keep existing index
BATCH_SIZE = 5000  # must be <= Chroma max batch size (yours: 5461)

if RESET_COLLECTION:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print('Deleted existing collection:', COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
)

ids_all = [d['id'] for d in docs]
docs_all = [d['text'] for d in docs]

for start in range(0, len(docs_all), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(docs_all))
    collection.add(
        ids=ids_all[start:end],
        documents=docs_all[start:end],
    )
    print(f'Indexed {end}/{len(docs_all)}')

print('Indexed total:', collection.count())


Indexed 5000/46725
Indexed 10000/46725
Indexed 15000/46725
Indexed 20000/46725
Indexed 25000/46725
Indexed 30000/46725
Indexed 35000/46725
Indexed 40000/46725
Indexed 45000/46725
Indexed 46725/46725
Indexed total: 46725


In [7]:
def call_llama_chat(
    prompt: str,
    model: str = 'llama3.1:8b',
    base_url: str = 'http://localhost:11434',
) -> str:
    url = f'{base_url}/api/chat'
    resp = requests.post(
        url,
        json={
            'model': model,
            'messages': [{'role': 'user', 'content': prompt}],
            'stream': False,
        },
    )
    resp.raise_for_status()
    return resp.json()['message']['content']


def rag_answer(question: str, top_k: int = 16) -> str:
    results = collection.query(query_texts=[question], n_results=top_k)
    context = '\n'.join(results['documents'][0])

    prompt = f"""
You answer questions using ONLY the facts in the context.
If the answer is not present, say you don't know.

Context (KG triples):
{context}

Question: {question}
Answer:
""".strip()

    return call_llama_chat(prompt)


In [8]:
# Example query
question = 'Which team_id is RB Leipzig associated with?'
print(rag_answer(question, top_k=20))


RB Leipzig is associated with the following team_ids:
1. Fabio Carvalho
2. Xaver Schlager
3. Willi Orban
4. Benjamin Henrichs
5. Xavi Simons
6. Janis Blaswich
7. Nicolas Seiwald
8. Kevin Kampl
9. Mohamed Simakan
